# Python Use Case Examples

In [ ]:
### Simulation of a coin flip 10,000 times

flips = 0

for i in range(10000)

####        Function || range():   

The Python range() function generates an immutable sequence of numbers, which is most commonly used for looping a specific number of times in for loops. It is highly memory-efficient because it calculates numbers on the fly rather than creating a full list in memory.

## --------------------------------------------------

# Packages and Variables

In [2]:
%pip install databento
import os 
from dotenv import load_dotenv
import logging
import databento as db
import asyncio
import pandas as pd
from pathlib import Path

load_dotenv() # used for local run 

import config
logging.basicConfig(level=logging.INFO)

app_key = config.DATABENTO_API_KEY
# # discontinued because of 7-day refresh 
# app_key = config.SCHWAB_API_KEY
# app_secret = config.SCHWAB_APP_SECRET



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:

def _client():
    key = getattr(config, "DATABENTO_API_KEY", "") or os.getenv("DATABENTO_API_KEY", "")
    if not key:
        raise RuntimeError("DATABENTO_API_KEY not set (config.py or .env).")
    return key

# # # # ### test databento connection
# client = db.Historical(_client())

# data = client.timeseries.get_range(
#     dataset=config.DATASET,
#     schema=config.SCHEMA,
#     start='2026-05-01T00:00',
#     end='2026-05-30T20:00',
#     limit=100,
# )

# data.replay(print)


### final check

In [ ]:
# # # Dataframe format handover check from load_all_timeframes
## Claude corrected validation cell
def check_alignment(name, df, rule):
    problems = []
    floor_min = df_1m.index.min().floor(rule)
    if df.index.min() != floor_min:
        problems.append(f"{name}: min {df.index.min()} != expected bucket start {floor_min}")
    if df.index.max() > df_1m.index.max():
        problems.append(f"{name}: max {df.index.max()} exceeds df_1m max {df_1m.index.max()}")
    return problems

all_problems = []
for name, df in [("1m", df_1m), ("5m", df_5m), ("15m", df_15m), ("1h", df_1h)]:
    all_problems += check_frame(name, df)

all_problems += check_alignment("5m",  df_5m,  "5min")
all_problems += check_alignment("15m", df_15m, "15min")
all_problems += check_alignment("1h",  df_1h,  "1h")

print("problems:", all_problems if all_problems else "none ✅")

problems: none ✅


In [2]:
print("1m  min:", df_1m.index.min())
print("5m  min:", df_5m.index.min())
print("15m min:", df_15m.index.min())
print("1h  min:", df_1h.index.min())

1m  min: 2026-04-27 01:54:00+00:00
5m  min: 2026-04-27 01:50:00+00:00
15m min: 2026-04-27 01:45:00+00:00
1h  min: 2026-04-27 01:00:00+00:00


In [1]:
### Revised Databento datafeed check - this threw an assertion error

import sys
sys.path.insert(0, "/Users/efiles/dev/traderpost analysis/etrade_scratch_clone")

import config
from data_feed import load_all_timeframes

df_1m, df_5m, df_15m, df_1h = load_all_timeframes(days=config.DATA_DAYS)

# ── The exact contract signals.py's run_signal_check() relies on ──
def check_frame(name, df):
    problems = []
    if list(df.columns) != ["open", "high", "low", "close"]:
        problems.append(f"{name}: unexpected columns {list(df.columns)}")
    if str(df.index.tz) != "UTC":
        problems.append(f"{name}: index tz is {df.index.tz}, expected UTC")
    if not df.index.is_monotonic_increasing:
        problems.append(f"{name}: index not sorted")
    if df.index.duplicated().any():
        problems.append(f"{name}: {df.index.duplicated().sum()} duplicate timestamps")
    if df.isna().any().any():
        problems.append(f"{name}: contains NaN — resample/dropna gap")
    if (df["high"] < df["low"]).any():
        problems.append(f"{name}: high < low on some bars — bad data")
    if (df["high"] < df[["open","close"]].max(axis=1)).any() or \
       (df["low"]  > df[["open","close"]].min(axis=1)).any():
        problems.append(f"{name}: OHLC internally inconsistent")
    return problems

all_problems = []
for name, df in [("1m", df_1m), ("5m", df_5m), ("15m", df_15m), ("1h", df_1h)]:
    print(f"{name:4} {df.shape} | {df.index.min()} -> {df.index.max()} | tz={df.index.tz}")
    all_problems += check_frame(name, df)

# ── Cross-timeframe consistency: 5m/15m/1h are resampled FROM df_1m, so their
#    span must sit inside df_1m's span, and bar counts should roughly divide. ──
assert df_5m.index.min()  >= df_1m.index.min() and df_5m.index.max()  <= df_1m.index.max()
assert df_15m.index.min() >= df_1m.index.min() and df_15m.index.max() <= df_1m.index.max()
assert df_1h.index.min()  >= df_1m.index.min() and df_1h.index.max()  <= df_1m.index.max()

# ── Enough history for signals.py's actual lookbacks (this is the assumption
#    the DATA_DAYS bump to 130 exists to satisfy) ──
span_days = (df_1m.index.max() - df_1m.index.min()).days
if span_days < config.TREND_WINDOW:
    all_problems.append(f"only {span_days}d of 1m history — TREND_WINDOW needs {config.TREND_WINDOW}d")

print("\nproblems:", all_problems if all_problems else "none ✅")

/Users/efiles/dev/traderpost analysis/etrade_scratch_clone/data_feed.py:67: BentoWarning: The streaming request contained one or more days which have reduced quality: 2026-05-24 (degraded), 2026-07-30 (degraded), 2026-08-29 (degraded). See: https://databento.com/docs/api-reference-historical/metadata/metadata-get-dataset-condition
  store = db.Historical(_client()).timeseries.get_range(


1m     | 127585 bars | 2026-04-27 to 2026-09-03 | range 48607-54885 | 129 days | 18 weeks | dupes removed: 0
OK Databento | 1m:127585 5m:25764 15m:8589 1h:2148
1m   (127585, 4) | 2026-04-27 01:54:00+00:00 -> 2026-09-03 22:52:00+00:00 | tz=UTC
5m   (25764, 4) | 2026-04-27 01:50:00+00:00 -> 2026-09-03 22:50:00+00:00 | tz=UTC
15m  (8589, 4) | 2026-04-27 01:45:00+00:00 -> 2026-09-03 22:45:00+00:00 | tz=UTC
1h   (2148, 4) | 2026-04-27 01:00:00+00:00 -> 2026-09-03 22:00:00+00:00 | tz=UTC


AssertionError: 

In [2]:
### Live API connection test
%pip install nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
import sys, time
sys.path.insert(0, "/Users/efiles/dev/traderpost analysis/etrade_scratch_clone")

import pandas as pd
import config
from data_feed import pull_via_live_replay, _pull_1m

# Small window first — last 20 minutes. Cheap, fast, easy to eyeball.
start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(minutes=20)

t0 = time.time()
try:
    df_replay = pull_via_live_replay(start, timeout=30)
    elapsed = time.time() - t0
    print(f"OK — {len(df_replay)} bars in {elapsed:.1f}s")
    print(df_replay)
except TimeoutError as e:
    print(f"TIMED OUT after {time.time()-t0:.1f}s: {e}")
except Exception as e:
    print(f"FAILED after {time.time()-t0:.1f}s: {type(e).__name__}: {e}")


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
replay |     18 bars | 2026-09-04 to 2026-09-04 | range 53754-53767 | 0 days | 0 weeks | dupes removed: 0
OK — 18 bars in 0.8s
                              open     high      low    close
datetime                                                     
2026-09-04 01:50:00+00:00  53760.0  53761.0  53756.0  53758.0
2026-09-04 01:52:00+00:00  53755.0  53759.0  53755.0  53759.0
2026-09-04 01:53:00+00:00  53759.0  53760.0  53757.0  53758.0
2026-09-04 01:54:00+00:00  53758.0  53759.0  53757.0  53757.0
2026-09-04 01:55:00+00:00  53756.0  53756.0  53754.0  53754.0
2026-09-04 01:56:00+00:00  53756.0  53757.0  53754.0  53757.0
2026-09-04 01:57:00+00:00  53755.0  53761.0  53755.0  53761.0
2026-09-04 01:58:00+00:00  53759.0  53759.0  53759.0  53759.0
2026-09-04 01:59:00+00:00  53761.0  53761.0  53759.0  53759.0
2026-09-04 02:

In [3]:
gaps = df_replay.index.to_series().diff()
print(gaps[gaps > pd.Timedelta(minutes=1)])

datetime
2026-09-04 01:52:00+00:00   0 days 00:02:00
Name: datetime, dtype: timedelta64[ns]


In [4]:
lag = pd.Timestamp.now(tz="UTC") - df_replay.index.max()
print(f"Replay data is {lag.total_seconds():.0f}s behind now")



Replay data is 1418s behind now


BentoClientError: 422 data_time_range_start_on_or_after_end
Invalid time range query, `start` 2026-09-04 01:49:00.490424+00:00 cannot be on or after `end` 2026-09-03 23:31:37.989325+00:00.
documentation: https://databento.com/docs/standards-and-conventions/common-fields-enums-types

In [6]:
older_start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=4)
older_end   = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=3, minutes=30)

df_replay_old = pull_via_live_replay(older_start, timeout=30)
df_hist_old   = _pull_1m(start=older_start, end=older_end)

overlap = df_replay_old.index.intersection(df_hist_old.index)
print(f"{len(overlap)} overlapping timestamps")
if len(overlap):
    diff = (df_replay_old.loc[overlap] - df_hist_old.loc[overlap]).abs().max()
    print("max abs diff per column:", diff.to_dict())

replay |    217 bars | 2026-09-03 to 2026-09-04 | range 53727-53773 | 1 days | 0 weeks | dupes removed: 0
1m     |     26 bars | 2026-09-03 to 2026-09-03 | range 53732-53747 | 0 days | 0 weeks | dupes removed: 0
26 overlapping timestamps
max abs diff per column: {'open': 0.0, 'high': 0.0, 'low': 0.0, 'close': 0.0}


In [7]:
start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(minutes=20)
df_replay = pull_via_live_replay(start, timeout=30)
lag = pd.Timestamp.now(tz="UTC") - df_replay.index.max()
print(f"Replay data is {lag.total_seconds():.0f}s behind now")

replay |     15 bars | 2026-09-04 to 2026-09-04 | range 53753-53769 | 0 days | 0 weeks | dupes removed: 0
Replay data is 73s behind now


In [ ]:
df_hist = _pull_1m(start=start)
overlap = df_replay.index.intersection(df_hist.index)
print(f"{len(overlap)} overlapping timestamps")
if len(overlap):
    diff = (df_replay.loc[overlap] - df_hist.loc[overlap]).abs().max()
    print("max abs diff per column:", diff.to_dict())

# Dev Cells

In [4]:
def _clean_1m(df, label="1m"):
    """Raw Databento ohlcv-1m -> tz-aware UTC 'datetime' index, spreads & dupes
    removed, OHLC only — the SAME cleaning the backtest CSV got, so live and
    backtest bars are treated identically."""
    if df.empty:
        print(f"{label:6} |      0 bars | (no data returned)")
        return pd.DataFrame(columns=["open", "high", "low", "close"])
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "datetime"
    df = df.dropna().sort_index()
    if "symbol" in df.columns:                                       # drop roll-spread rows
        df = df[~df["symbol"].astype(str).str.contains("-")]
    dupes = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")]
    df = df[["open", "high", "low", "close"]].copy()
    d0, d1 = df.index[0].date(), df.index[-1].date()
    days = (d1 - d0).days
    print(f"{label:6} | {len(df):>6} bars | {d0} to {d1} | "
          f"range {df['low'].min():.0f}-{df['high'].max():.0f} | "
          f"{days} days | {days // 7} weeks | dupes removed: {dupes}")
    return df

def _safe_end(end=None):
    """End-time capped safely in the past (Historical API rejects the recent edge)."""
    return end if end is not None else pd.Timestamp.now(tz="UTC") - config.DATA_LAG

# # ### test the value return for API data lag
# time.tz_convert("UTC")
# time = (_safe_end(end=None))
# print(time)
def _pull_1m(start, end=None):
    """Low-level Databento ohlcv-1m pull -> cleaned 1m DataFrame."""
    store = db.Historical(_client()).timeseries.get_range(
        dataset=config.DATASET, symbols=[config.SYMBOL], schema=config.SCHEMA, stype_in=config.STYPE_IN,
        start=start, end=_safe_end(end),
    )
    return _clean_1m(store.to_df())


In [5]:
# -- HISTORICAL PULL -------------------------------------------

import matplotlib.pyplot as plt

def load_all_timeframes(days=config.DATA_DAYS, cache_path=None):
    """Batch snapshot for signals.py. Returns (df_1m, df_5m, df_15m, df_1h).

    cache_path=None : fresh pull of the full window (simplest).
    cache_path set  : incremental — read cache, pull only the gap since the last
                      cached minute, append, dedupe, trim, rewrite. Saves credits.
                      Use a DEDICATED cache file, not the backtest base CSV.
    """
    cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)

    if cache_path and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cached["datetime"] = pd.to_datetime(cached["datetime"], utc=True)
        cached = cached.set_index("datetime").sort_index()
        fresh = _pull_1m(start=cached.index.max())                  # only the gap
        df_1m = pd.concat([cached[["open", "high", "low", "close"]], fresh])
        df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()
    else:
        df_1m = _pull_1m(start=cutoff)

    df_1m = df_1m[df_1m.index >= cutoff]                            # trim to rolling window
    if cache_path:
        df_1m.reset_index().to_csv(cache_path, index=False)

    df_5m  = resample_ohlc(df_1m, "5min")
    df_15m = resample_ohlc(df_1m, "15min")
    df_1h  = resample_ohlc(df_1m, "1h")
    print(f"OK Databento | 1m:{len(df_1m)} 5m:{len(df_5m)} 15m:{len(df_15m)} 1h:{len(df_1h)}")
    return df_1m, df_5m, df_15m, df_1h

def estimate_databento_cost(days=config.DATA_DAYS):
    """$ cost of the `days`-day 1m pull, WITHOUT spending credits."""
    start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)
    return db.Historical(_client()).metadata.get_cost(
        dataset=config.DATASET, symbols=[config.SYMBOL], schema=config.SCHEMA, stype_in=config.STYPE_IN,
        start=start, end=_safe_end(),
    )  

class _ReplayCollector:
    """Buffers 1m bars from a db.Live intraday replay and stops the
    session as soon as the historical portion finishes."""

    def __init__(self, client):
        self._client = client
        self.bars = []

    def __call__(self, record):
        if isinstance(record, db.OHLCVMsg):
            self.bars.append({
                "datetime": record.pretty_ts_event,
                "open": record.pretty_open, "high": record.pretty_high,
                "low": record.pretty_low, "close": record.pretty_close,
            })
        elif isinstance(record, db.SystemMsg) and record.code == db.SystemCode.REPLAY_COMPLETED:
            self._client.stop()

    def to_df(self):
        return pd.DataFrame(self.bars).set_index("datetime") if self.bars else pd.DataFrame()


In [7]:
%pip install asyncio
"""Dry-run validation of the revised _clean_1m + replay-collector design
against the REAL databento_dbn record classes (no network, no API key).
"""
import databento as db
import databento_dbn as dbn
import pandas as pd

def _clean_1m(df, label="1m"):
    """Raw Databento ohlcv-1m -> tz-aware UTC 'datetime' index, spreads & dupes
    removed, OHLC only — the SAME cleaning the backtest CSV got, so live and
    backtest bars are treated identically."""
    if df.empty:
        print(f"{label:6} |      0 bars | (no data returned)")
        return pd.DataFrame(columns=["open", "high", "low", "close"])
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "datetime"
    df = df.dropna().sort_index()
    if "symbol" in df.columns:
        df = df[~df["symbol"].astype(str).str.contains("-")]
    dupes = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")]
    df = df[["open", "high", "low", "close"]].copy()
    d0, d1 = df.index[0].date(), df.index[-1].date()
    days = (d1 - d0).days
    print(f"{label:6} | {len(df):>6} bars | {d0} to {d1} | "
          f"range {df['low'].min():.0f}-{df['high'].max():.0f} | "
          f"{days} days | {days // 7} weeks | dupes removed: {dupes}")
    return df

def load_all_timeframes(days=config.DATA_DAYS, cache_path=None):
    """Batch snapshot for signals.py. Returns (df_1m, df_5m, df_15m, df_1h).

    cache_path=None : fresh pull of the full window (simplest).
    cache_path set  : incremental — read cache, pull only the gap since the last
                      cached minute, append, dedupe, trim, rewrite. Saves credits.
                      Use a DEDICATED cache file, not the backtest base CSV.
    """
    cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)

    if cache_path and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cached["datetime"] = pd.to_datetime(cached["datetime"], utc=True)
        cached = cached.set_index("datetime").sort_index()
        fresh = _pull_1m(start=cached.index.max())                  # only the gap
        df_1m = pd.concat([cached[["open", "high", "low", "close"]], fresh])
        df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()
    else:
        df_1m = _pull_1m(start=cutoff)

    df_1m = df_1m[df_1m.index >= cutoff]                            # trim to rolling window
    if cache_path:
        df_1m.reset_index().to_csv(cache_path, index=False)

    df_5m  = resample_ohlc(df_1m, "5min")
    df_15m = resample_ohlc(df_1m, "15min")
    df_1h  = resample_ohlc(df_1m, "1h")
    print(f"OK Databento | 1m:{len(df_1m)} 5m:{len(df_5m)} 15m:{len(df_15m)} 1h:{len(df_1h)}")
    return df_1m, df_5m, df_15m, df_1h

def estimate_databento_cost(days=config.DATA_DAYS):
    """$ cost of the `days`-day 1m pull, WITHOUT spending credits."""
    start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)
    return db.Historical(_client()).metadata.get_cost(
        dataset=config.DATASET, symbols=[config.SYMBOL], schema=config.SCHEMA, stype_in=config.STYPE_IN,
        start=start, end=_safe_end(),
    )

class _ReplayCollector:
    """Buffers 1m bars from a db.Live intraday replay and stops the
    session as soon as the historical portion finishes."""

    def __init__(self, client):
        self._client = client
        self.bars = []

    def __call__(self, record):
        if isinstance(record, db.OHLCVMsg):
            self.bars.append({
                "datetime": record.pretty_ts_event,
                "open": record.pretty_open, "high": record.pretty_high,
                "low": record.pretty_low, "close": record.pretty_close,
            })
        elif isinstance(record, db.SystemMsg) and record.code == db.SystemCode.REPLAY_COMPLETED:
            self._client.stop()

    def to_df(self):
        return pd.DataFrame(self.bars).set_index("datetime") if self.bars else pd.DataFrame()


class _FakeClient:
    """Stand-in for db.Live — just needs a .stop() to prove the collector calls it."""
    def __init__(self):
        self.stopped = False

    def stop(self):
        self.stopped = True


def make_ohlcv(ts_ns, o, h, l, c, instrument_id=42004247):
    return dbn.OHLCVMsg(
        rtype=32, publisher_id=1, instrument_id=instrument_id,
        ts_event=ts_ns, open=int(o * 1e9), high=int(h * 1e9),
        low=int(l * 1e9), close=int(c * 1e9), volume=100,
    )

def make_replay_completed():
    return dbn.SystemMsg(ts_event=0, msg="Finished ohlcv-1m replay", code=dbn.SystemCode.REPLAY_COMPLETED)

def make_subscription_ack():
    return dbn.SystemMsg(ts_event=0, msg="Subscription request 0 succeeded", code=dbn.SystemCode.SUBSCRIPTION_ACK)



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ---- Test 1: normal window with a few bars, system msgs interleaved, one dupe ----
print("=== Test 1: bars + system msgs + one duplicate timestamp ===")
base = pd.Timestamp("2026-09-03 13:30:00", tz="UTC").value  # ns since epoch
client = _FakeClient()
collector = _ReplayCollector(client)

collector(make_subscription_ack())                          # should be ignored, no crash
collector(make_ohlcv(base + 0 * 60_000_000_000, 100, 101, 99, 100.5))
collector(make_ohlcv(base + 1 * 60_000_000_000, 100.5, 102, 100, 101.5))
collector(make_ohlcv(base + 1 * 60_000_000_000, 100.5, 102.2, 100, 101.7))  # dupe minute, should win (keep='last')
assert client.stopped is False, "client.stop() should NOT have been called yet"
collector(make_replay_completed())
assert client.stopped is True, "client.stop() should be called on REPLAY_COMPLETED"

df = collector.to_df()
print(df)
result = _clean_1m(df, label="test")
print(result)
assert len(result) == 2, f"expected 2 bars after dedupe, got {len(result)}"
assert result.iloc[1]["high"] == 102.2, "dupe should keep the LAST record, not the first"
print("Test 1 PASSED\n")

# ---- Test 2: empty window (no OHLCVMsg records at all, e.g. market closed) ----
print("=== Test 2: empty window ===")
client2 = _FakeClient()
collector2 = _ReplayCollector(client2)
collector2(make_subscription_ack())
collector2(make_replay_completed())
assert client2.stopped is True
df2 = collector2.to_df()
result2 = _clean_1m(df2, label="empty")
assert result2.empty and list(result2.columns) == ["open", "high", "low", "close"]
print("Test 2 PASSED\n")

# ---- Test 3: 1h fetch -> pandas slice to derive 15m window (no second connection) ----
print("=== Test 3: single fetch, slice for narrower window ===")
client3 = _FakeClient()
collector3 = _ReplayCollector(client3)
now = pd.Timestamp("2026-09-03 14:00:00", tz="UTC")
for i in range(60, 0, -5):                                   # bars every 5 min for the last hour
    ts = now - pd.Timedelta(minutes=i)
    collector3(make_ohlcv(ts.value, 100 + i, 101 + i, 99 + i, 100.5 + i))
collector3(make_replay_completed())
df_1h = _clean_1m(collector3.to_df(), label="1h")
cutoff = now - pd.Timedelta(minutes=15)
df_15m = df_1h[df_1h.index >= cutoff]
print(f"1h bars: {len(df_1h)}, 15m slice: {len(df_15m)}")
assert len(df_15m) < len(df_1h), "15m slice should be a strict subset of the 1h window"
assert (df_15m.index >= cutoff).all()
print("Test 3 PASSED\n")

print("ALL TESTS PASSED")

# # # # data = client.timeseries.get_range(
# # # #     dataset="GLBX.MDP3",
# # # #     schema="ohlcv-1d",
# # # #     stype_in="continuous",
# # # #     symbols=config.SYMBOL,
# # # #     start='2026-05-01T00:00',
# # # #     end='2026-05-30T20:00',
# # # # )

# # # # # # ### test returned data by charting data
# # # # df = data.to_df()
# # # # df.groupby("symbol")["close"].plot(
# # # #     xlabel="Date",
# # # #     ylabel="Price",
# # # # )

# # # # plt.legend()
# # # # plt.show()

# Live Feed API Data

4502:class OHLCVMsg:
4503-    """
4504-    Open, high, low, close, and volume. The record of the following schemas:
4505-    - OHLCV-1s
4506-    - OHLCV-1m
4507-    - OHLCV-1h
4508-    - OHLCV-1d
4509-    - OHLCV-eod
4510-
4511-    """
4512-
4513-    size_hint: ClassVar[int]
4514-    _dtypes: ClassVar[list[tuple[str, str]]]
4515-    _hidden_fields: ClassVar[list[str]]
4516-    _price_fields: ClassVar[list[str]]
4517-    _ordered_fields: ClassVar[list[str]]

In [ ]:
import databento as db
import inspect

print('=== Live class ===')
print('has __enter__:', hasattr(db.Live, '__enter__'))
print('has __exit__:', hasattr(db.Live, '__exit__'))
print()
print('=== block_for_close signature ===')
print(inspect.signature(db.Live.block_for_close))
print(db.Live.block_for_close.__doc__)


In [ ]:
import databento as db
import databento_dbn as dbn
import inspect

print('=== module-level exports check ===')
print('OHLCVMsg' in dir(db), 'SystemMsg' in dir(db), 'SymbolMappingMsg' in dir(db))
print(db.OHLCVMsg is dbn.OHLCVMsg)
print()
print('=== SystemMsg fields ===')
print([a for a in dir(dbn.SystemMsg) if not a.startswith('_')])
print()
print('=== OHLCVMsg fields ===')
print([a for a in dir(dbn.OHLCVMsg) if not a.startswith('_')])

In [ ]:
import databento as db
import databento_dbn as dbn

print('=== SystemCode enum? ===')
print(hasattr(dbn, 'SystemCode'), [e for e in dir(dbn.SystemCode) if not e.startswith('_')] if hasattr(dbn, 'SystemCode') else None)
print()
print('=== code property type ===')
print(inspect.signature(dbn.SystemMsg.code.fget) if hasattr(dbn.SystemMsg.code, 'fget') else dbn.SystemMsg.code)
print(dbn.SystemMsg.code.__doc__)
print()
print('=== pretty_open property ===')
print(dbn.OHLCVMsg.pretty_open.__doc__)
print(dbn.OHLCVMsg.open.__doc__)

import databento_dbn as dbn, inspect
print(dbn.SystemMsg.code.__doc__)
print()
print(dbn.OHLCVMsg.open.__doc__)
print()
print(dbn.OHLCVMsg.pretty_open.__doc__)


In [ ]:
import databento_dbn as dbn
import pandas as pd

rec = dbn.OHLCVMsg(
    rtype=32, publisher_id=1, instrument_id=42004247,
    ts_event=1735689600_000000000,
    open=5000_000000000, high=5010_000000000, low=4990_000000000, close=5005_000000000,
    volume=100,
)
print('pretty_ts_event:', rec.pretty_ts_event, type(rec.pretty_ts_event), rec.pretty_ts_event.tzinfo)
print('pretty_open:', rec.pretty_open, type(rec.pretty_open))
print('open/1e9:', rec.open / 1e9)

Initialize self.  See help(type(self)) for accurate signature.
7146:class SystemMsg:
7147-    """
7148-    A non-error message from the Databento Live Subscription Gateway (LSG). Also used
7149-    for heartbeating.
7150-
7151-    """
7152-
7153-    size_hint: ClassVar[int]
7154-    _dtypes: ClassVar[list[tuple[str, str]]]
7155-    _hidden_fields: ClassVar[list[str]]
7156-    _price_fields: ClassVar[list[str]]
7157-    _ordered_fields: ClassVar[list[str]]
7158-    _timestamp_fields: ClassVar[list[str]]
7159-
7160-    def __init__(
7161-        self, ts_event: int, msg: str, code: SystemCode | None = None, ts_out: int = UNDEF_TIMESTAMP
7162-    ) -> None: ...
7163-    def __bytes__(self) -> bytes: ...
7164-    def record_size(self) -> int:
7165-        """
7166-        Return the size of the record in bytes

In [ ]:
import databento_dbn as dbn
import databento as db

sm = dbn.SystemMsg(ts_event=0, msg='Finished ohlcv-1m replay', code=dbn.SystemCode.REPLAY_COMPLETED)
print(sm.code, sm.code == dbn.SystemCode.REPLAY_COMPLETED, repr(sm.msg))
print(isinstance(sm, db.SystemMsg))
print()
import inspect
print([m for m in dir(db.Live) if 'callback' in m.lower() or 'stream' in m.lower() or 'stop' in m.lower() or 'terminate' in m.lower() or 'exception' in m.lower()])
print(inspect.signature(db.Live.add_callback))

In [ ]:
import databento as db
import inspect
print('stop:', inspect.signature(db.Live.stop))
print(db.Live.stop.__doc__)
print()
print('terminate:', inspect.signature(db.Live.terminate))
print(db.Live.terminate.__doc__)
print()
print('add_reconnect_callback:', inspect.signature(db.Live.add_reconnect_callback))

In [ ]:
%pip install databento-dbn 
## Live Feed - Databento API
import databento as db
import pandas as pd
import databento_dbn as dbn

def _clean_1m(df, label="1m"):
    """Raw Databento ohlcv-1m -> tz-aware UTC 'datetime' index, spreads & dupes
    removed, OHLC only — the SAME cleaning the backtest CSV got, so live and
    backtest bars are treated identically."""
    if df.empty:
        print(f"{label:6} |      0 bars | (no data returned)")
        return pd.DataFrame(columns=["open", "high", "low", "close"])
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "datetime"
    df = df.dropna().sort_index()
    if "symbol" in df.columns:
        df = df[~df["symbol"].astype(str).str.contains("-")]
    dupes = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")]
    df = df[["open", "high", "low", "close"]].copy()
    d0, d1 = df.index[0].date(), df.index[-1].date()
    days = (d1 - d0).days
    print(f"{label:6} | {len(df):>6} bars | {d0} to {d1} | "
          f"range {df['low'].min():.0f}-{df['high'].max():.0f} | "
          f"{days} days | {days // 7} weeks | dupes removed: {dupes}")
    return df

def resample_ohlc(df, rule):
    return df.resample(rule).agg(
        {"open": "first", "high": "max", "low": "min", "close": "last"}
    ).dropna()

def _pull_1m(start, end=None):
    """Low-level Databento ohlcv-1m pull -> cleaned 1m DataFrame."""
    store = db.Historical(_client()).timeseries.get_range(
        dataset=DATASET, symbols=[SYMBOL], schema=SCHEMA, stype_in=STYPE_IN,
        start=start, end=_safe_end(end),
    )
    return _clean_1m(store.to_df())

# -- CREDENTIALS -----------------------------------------------
def _client():
    key = getattr(config, "DATABENTO_API_KEY", "") or os.getenv("DATABENTO_API_KEY", "")
    if not key:
        raise RuntimeError("DATABENTO_API_KEY not set (config.py or .env).")
    return key

# # # ### test databento connection
client = db.Live(_client())

data = client.subscribe(
    dataset=config.DATASET,
    schema=config.SCHEMA,
    stype_in=config.STYPE_IN,
    # start='2026-05-01T00:00',     --- not needed for live calls
    # end='2026-05-30T20:00',       --- not needed for live calls
    symbols=config.SYMBOL,
    start=pd.Timestamp.now(tz="UTC")-config.DATA_LAG,
    #end=pd.Timestamp.now(tz="UTC"),
)

# Now, we will open a file for writing and start streaming
client.add_callback(_client())

client.start()

# keep live feed open for 5 seconds
client.block_for_close(timeout=5)

# def _collect(record):
#     if isinstance(record, db.OHLCVMsg):            # ignore SystemMsg/SymbolMappingMsg
#         bars.append({
#             "datetime": pd.Timestamp(record.ts_event, unit="ns", tz="UTC"),
#             "open": record.open / 1e9, "high": record.high / 1e9,
#             "low": record.low / 1e9, "close": record.close / 1e9,
#         })
# def load_all_timeframes(data):
#     """Batch snapshot for signals.py. Returns (df_1m, df_5m, df_15m, df_1h).

#     cache_path=None : fresh pull of the full window (simplest).
#     cache_path set  : incremental — read cache, pull only the gap since the last
#                       cached minute, append, dedupe, trim, rewrite. Saves credits.
#                       Use a DEDICATED cache file, not the backtest base CSV.
#     """
#     cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)

#     if cache_path and os.path.exists(cache_path):
#         cached = pd.read_csv(cache_path)
#         cached["datetime"] = pd.to_datetime(cached["datetime"], utc=True)
#         cached = cached.set_index("datetime").sort_index()
#         fresh = _pull_1m(start=cached.index.max())                  # only the gap
#         df_1m = pd.concat([cached[["open", "high", "low", "close"]], fresh])
#         df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()
#     else:
#         df_1m = _pull_1m(start=cutoff)

#     df_1m = df_1m[df_1m.index >= cutoff]                            # trim to rolling window
#     if cache_path:
#         df_1m.reset_index().to_csv(cache_path, index=False)

#     df_5m  = resample_ohlc(df_1m, "5min")
#     df_15m = resample_ohlc(df_1m, "15min")
#     df_1h  = resample_ohlc(df_1m, "1h")
#     print(f"OK Databento | 1m:{len(df_1m)} 5m:{len(df_5m)} 15m:{len(df_15m)} 1h:{len(df_1h)}")
#     return df_1m, df_5m, df_15m, df_1h


In [ ]:
"""Dry-run validation of the revised _clean_1m + replay-collector design
against the REAL databento_dbn record classes (no network, no API key).
"""
from __future__ import annotations

import os 
from dotenv import load_dotenv
from pathlib import Path
import logging
import asyncio
import databento as db
import databento_dbn as dbn
import pandas as pd

load_dotenv() # used for local run 

import config
logging.basicConfig(level=logging.INFO)

app_key = config.DATABENTO_API_KEY
# app_key = config.SCHWAB_API_KEY
# app_secret = config.SCHWAB_APP_SECRET
def _client():
    key = getattr(config, "DATABENTO_API_KEY", "") or os.getenv("DATABENTO_API_KEY", "")
    if not key:
        raise RuntimeError("DATABENTO_API_KEY not set (config.py or .env).")
    return key

#### --- CLEANING / RESAMPLE function, what do you want the data to look like ---
# Shared by db.Historical object + db.Live object 
# Here the data cleaning function is here, alot of understanding of the data process here
# Study the ways the data is manipulated and compare it to SQL functions at working
####

def _clean_1m(df, label="1m"):
    """Raw Databento ohlcv-1m -> tz-aware UTC 'datetime' index, spreads & dupes
    removed, OHLC only — the SAME cleaning the backtest CSV got, so live and
    backtest bars are treated identically."""
    if df.empty:
        print(f"{label:6} |      0 bars | (no data returned)")
        return pd.DataFrame(columns=["open", "high", "low", "close"])
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "datetime"
    df = df.dropna().sort_index()
    if "symbol" in df.columns:
        df = df[~df["symbol"].astype(str).str.contains("-")]
    dupes = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")]
    df = df[["open", "high", "low", "close"]].copy()
    d0, d1 = df.index[0].date(), df.index[-1].date()
    days = (d1 - d0).days
    print(f"{label:6} | {len(df):>6} bars | {d0} to {d1} | "
          f"range {df['low'].min():.0f}-{df['high'].max():.0f} | "
          f"{days} days | {days // 7} weeks | dupes removed: {dupes}")
    return df


def resample_ohlc(df, rule):
    return df.resample(rule).agg(
        {"open": "first", "high": "max", "low": "min", "close": "last"}
    ).dropna()

#### --- api time gap stitching functions --- 
def _safe_end(end=None): ### to account for data lag
    """End-time capped safely in the past (Historical API rejects the recent edge)."""
    return end if end is not None else pd.Timestamp.now(tz="UTC") - DATA_LAG

def _pull_1m(start, end=None):
    """Low-level Databento ohlcv-1m pull -> cleaned 1m DataFrame."""
    store = db.Historical(_client()).timeseries.get_range(
        dataset=config.DATASET, symbols=[config.SYMBOL], schema=config.SCHEMA, stype_in=config.STYPE_IN,
        start=start, end=_safe_end(end),
    )
    return _clean_1m(store.to_df())
    # # # def get_range(
    # # #     self,
    # # #     dataset: Dataset | str,
    # # #     start: pd.Timestamp | datetime | date | str | int,
    # # #     end: pd.Timestamp | datetime | date | str | int | None = None,
    # # #     symbols: Iterable[str | int] | str | int | None = None,
    # # #     schema: Schema | str = "trades",
    # # #     stype_in: SType | str = "raw_symbol",
    # # #     stype_out: SType | str = "instrument_id",
    # # #     limit: int | None = None,
    # # #     path: PathLike[str] | str | None = None,
    # # # ) -> DBNStore:
    

def load_all_timeframes(days=config.DATA_DAYS, cache_path=None):
    """Batch snapshot for signals.py. Returns (df_1m, df_5m, df_15m, df_1h).

    cache_path=None : fresh pull of the full window (simplest).
    cache_path set  : incremental — read cache, pull only the gap since the last
                      cached minute, append, dedupe, trim, rewrite. Saves credits.
                      Use a DEDICATED cache file, not the backtest base CSV.
    """
    cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)

    if cache_path and os.path.exists(cache_path):
        cached = pd.read_csv(cache_path)
        cached["datetime"] = pd.to_datetime(cached["datetime"], utc=True)
        cached = cached.set_index("datetime").sort_index()
        fresh = _pull_1m(start=cached.index.max())                  # only the gap
        df_1m = pd.concat([cached[["open", "high", "low", "close"]], fresh])
        df_1m = df_1m[~df_1m.index.duplicated(keep="last")].sort_index()
    else:
        df_1m = _pull_1m(start=cutoff)

    df_1m = df_1m[df_1m.index >= cutoff]                            # trim to rolling window
    if cache_path:
        df_1m.reset_index().to_csv(cache_path, index=False)

    df_5m  = resample_ohlc(df_1m, "5min")
    df_15m = resample_ohlc(df_1m, "15min")
    df_1h  = resample_ohlc(df_1m, "1h")
    print(f"OK Databento | 1m:{len(df_1m)} 5m:{len(df_5m)} 15m:{len(df_15m)} 1h:{len(df_1h)}")
    return df_1m, df_5m, df_15m, df_1h

def estimate_databento_cost(days=config.DATA_DAYS):
    """$ cost of the `days`-day 1m pull, WITHOUT spending credits."""
    start = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=days)
    return db.Historical(_client()).metadata.get_cost(
        dataset=config.DATASET, symbols=[config.SYMBOL], schema=config.SCHEMA, stype_in=config.STYPE_IN,
        start=start, end=_safe_end(),
    )

class _ReplayCollector:
    """Buffers 1m bars from a db.Live intraday replay and stops the
    session as soon as the historical portion finishes."""

    def __init__(self, client):
        self._client = client
        self.bars = []

    def __call__(self, record):
        if isinstance(record, db.OHLCVMsg):
            self.bars.append({
                "datetime": record.pretty_ts_event,
                "open": record.pretty_open, "high": record.pretty_high,
                "low": record.pretty_low, "close": record.pretty_close,
            })
        elif isinstance(record, db.SystemMsg) and record.code == db.SystemCode.REPLAY_COMPLETED:
            self._client.stop()

    def to_df(self):
        return pd.DataFrame(self.bars).set_index("datetime") if self.bars else pd.DataFrame()


SyntaxError: invalid syntax (4236359989.py, line 58)